# Classification with an Academic Success Dataset

Playground Series - Season 6, Episode 7

(https://www.kaggle.com/competitions/playground-series-s6e7)

![image](./data/header.png)

__Overview__
Welcome to the 2026 Kaggle Playground Series! We plan to continue in the spirit of previous playgrounds, providing interesting and approachable datasets for our community to practice their machine learning skills, and anticipate a competition each month.

__Your Goal:__ Predicting student health risk.

__Dataset Description__
The dataset for this competition (both train and test) was inspired by the College Student Health Behavior Dataset. Feature distributions are close to, but not exactly the same, as the original.

__Files__
- train.csv - the training set, with ```health_condition``` as target
- test.csv -  the test set, used to predict the category for ```health_condition```
- sample_submission.csv - a sample submission file in the correct format

__Training__
Biometrics & Physical Metrics

- id — Unique identifier for each student.
- gender — Student gender (male, female, other).
- bmi — Body Mass Index.
- heart_rate — Measured heart rate in beats per minute (bpm).

Daily Habits & Activity

- step_count — Number of steps taken daily.
- calorie_expenditure — Total calories burned.
- exercise_duration — Time spent exercising (in minutes).
- physical_activity_level — Overall activity category (sedentary, moderate, active).
- water_intake — Daily water consumption (in liters/ml).
- diet_type — Dietary habit (veg, non-veg, balanced).

Sleep & Wellbeing

- sleep_duration — Hours of sleep per night.
- sleep_quality — Self-reported sleep quality (poor, average, good).
- stress_level — Reported stress level (low, medium, high).
- smoking_alcohol — Substance use habit (no, occasional, yes).

__Models__
- K-Nearest Neighboor Model 
- Gaussian Naive Bayes Model 
- Logistic Regressor 
- Support Vector Classification Model 
- Decision Tree Model 
- Random Forest Model 
- Linear Discriminant Analysis Model 
- Gradient Boosting Classifier Model 
- Neural Network CLassifier Model 
- X Gradient Boost Classifier
- Cat Boosting Classifier

In [ ]:
# Importing libraries
import numpy as np
import pandas as pd
import warnings as wn
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

In [ ]:
# Ignoring warnings
wn.filterwarnings('ignore')

In [8]:
# Defining constants

PLOT_FOLDER = "plots/"
DATA_PATH = "data/"
TESTING_DATA = DATA_PATH + "test.csv"
TRAINING_DATA = DATA_PATH + "train.csv"
SUBMISSION_DATA = DATA_PATH + "sample_submission.csv"

In [9]:
# Simplify the process of integrating large CSV

def reduce_mem_usage(df):
    start_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage of dataframe is {:.2f} MB'.format(start_mem))
    
    for col in df.columns:
        col_type = df[col].dtype
        
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print('Memory usage after optimization is: {:.2f} MB'.format(end_mem))
    print('Decreased by {:.1f}%'.format(100 * (start_mem - end_mem) / start_mem))
    
    return df

def import_data(file, **kwargs):
    df = pd.read_csv(file, parse_dates=True, keep_date_col=True, **kwargs)
    df = reduce_mem_usage(df)
    return df

In [12]:
# Reading the training data
train = import_data(TRAINING_DATA, index_col='id')

Memory usage of dataframe is 78.97 MB
Memory usage after optimization is: 51.33 MB
Decreased by 35.0%


In [16]:
# Displaying the first few rows of the training data
train.head(n=10)

,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender
id,,,,,,,,,,,,,,
0,unhealthy,5.218750,70.6250,25.656250,2174.0,1326.0,19.796875,1.860352,veg,high,average,sedentary,yes,female
1,at-risk,5.531250,71.3125,25.843750,1966.0,9888.0,49.906250,1.259766,non-veg,low,average,moderate,yes,other
2,unhealthy,5.289062,75.3750,24.546875,2688.0,14216.0,38.093750,1.599609,veg,high,poor,active,yes,male
3,unhealthy,4.699219,77.1875,23.125000,2630.0,7176.0,59.906250,2.019531,veg,high,average,active,occasional,female
4,at-risk,7.230469,73.3750,28.437500,2560.0,6584.0,46.000000,2.250000,veg,NaN,average,sedentary,NaN,male
5,at-risk,5.109375,82.8125,24.437500,2290.0,2134.0,29.203125,2.019531,veg,low,poor,sedentary,occasional,male
6,at-risk,8.210938,71.1250,24.140625,2724.0,12984.0,54.500000,2.009766,balanced,low,NaN,moderate,no,female
7,at-risk,7.468750,87.0000,23.515625,NaN,3096.0,37.500000,0.509766,balanced,low,average,sedentary,occasional,male
8,unhealthy,5.941406,75.3750,24.359375,NaN,12168.0,61.593750,1.559570,veg,high,average,active,no,other


In [ ]:
# Discribing the training data
train.describe()

,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake
count,614089.000000,682255.0000,676190.000000,637235.0,676172.0,683187.000000,646611.000000
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,0.000000,0.0000,0.000000,0.0,0.0,0.000000,0.000000
min,3.000000,50.0000,16.000000,1200.0,1002.0,0.000000,0.500000
25%,6.160156,69.3750,21.312500,2052.0,5388.0,29.203125,1.839844
50%,6.988281,75.1250,22.984375,2240.0,8856.0,39.406250,2.169922
75%,7.808594,80.6875,24.656250,2456.0,12112.0,49.406250,2.500000
max,10.000000,107.6875,34.812500,3580.0,15000.0,99.812500,4.718750


In [ ]:
# Columns informations
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 690088 entries, 0 to 690087
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   health_condition         690088 non-null  object 
 1   sleep_duration           614089 non-null  float16
 2   heart_rate               682255 non-null  float16
 3   bmi                      676190 non-null  float16
 4   calorie_expenditure      637235 non-null  float16
 5   step_count               676172 non-null  float16
 6   exercise_duration        683187 non-null  float16
 7   water_intake             646611 non-null  float16
 8   diet_type                683187 non-null  object 
 9   stress_level             607277 non-null  object 
 10  sleep_quality            631757 non-null  object 
 11  physical_activity_level  653467 non-null  object 
 12  smoking_alcohol          661506 non-null  object 
 13  gender                   668715 non-null  object 
dtypes: float1

In [29]:
# Number of unique values per non-numeric column
object_cols = train.select_dtypes(include=['object']).columns
for col in object_cols:
    print(f"{col}: {train[col].nunique()} unique values:  \t\t{train[col].unique()}")

health_condition: 3 unique values:  		['unhealthy' 'at-risk' 'fit']
diet_type: 3 unique values:  		['veg' 'non-veg' 'balanced' nan]
stress_level: 3 unique values:  		['high' 'low' nan 'medium']
sleep_quality: 3 unique values:  		['average' 'poor' nan 'good']
physical_activity_level: 3 unique values:  		['sedentary' 'moderate' 'active' nan]
smoking_alcohol: 3 unique values:  		['yes' 'occasional' nan 'no']
gender: 3 unique values:  		['female' 'other' 'male' nan]


In [31]:
columns = train.columns
for col in columns:
    if train[col].isnull().sum() > 0:
        print(f"{col}: {train[col].isnull().sum()} missing values ({train[col].isnull().sum()/len(train)*100:.2f}%)")

sleep_duration: 75999 missing values (11.01%)
heart_rate: 7833 missing values (1.14%)
bmi: 13898 missing values (2.01%)
calorie_expenditure: 52853 missing values (7.66%)
step_count: 13916 missing values (2.02%)
exercise_duration: 6901 missing values (1.00%)
water_intake: 43477 missing values (6.30%)
diet_type: 6901 missing values (1.00%)
stress_level: 82811 missing values (12.00%)
sleep_quality: 58331 missing values (8.45%)
physical_activity_level: 36621 missing values (5.31%)
smoking_alcohol: 28582 missing values (4.14%)
gender: 21373 missing values (3.10%)
